# IMPORT MODULES, INSTANTIATE CLASS

In [1]:
import glob
import os

import pandas as pd

from geoai.utils_geo.RasterOps import RasterOperations
from geoai.utils_ds.DataFrameOps import DataFrameOperations
from geoai.utils_geo.VectorOps import VectorOperations

raster_ops = RasterOperations()
df_ops = DataFrameOperations()
vector_ops = VectorOperations()

# SET GLOBAL VAR

In [2]:
RASTER_PATH = r"raster_files\rgbnirndvi.tif"
N_BANDS = 5
BAND_NAMES = ["BLUE", "GREEN", "RED", "NIR", "NDVI"]

# CLIP ROI SHAPEFILE TO RASTER

In [3]:
for shapefile in glob.glob(os.path.join("shapefiles", "*.shp")):
    roi_name = os.path.splitext(os.path.basename(shapefile))[0]
    output_raster_path = os.path.join("raster_files", f"{roi_name}.tif")
    vector_ops.clip_raster_with_shapefile(RASTER_PATH, shapefile, output_raster_path)

# MAKE A DF ON EVERY ROI RASTER

In [4]:
for roi_path in glob.glob(os.path.join("raster_files", "*.tif")):
    roi_name = os.path.splitext(os.path.basename(roi_path))[0]
    if roi_name in ["builtup", "trees", "water"]:
        df_bands = []
        array = raster_ops.raster_to_array(roi_path)
        for band_index, band_name in zip(range(N_BANDS), BAND_NAMES):
            flat = raster_ops.flatten_array(array, band_index)
            df = df_ops.convert_to_df(flat, band_name)
            df = df.loc[~(df==0).all(axis=1)] # remove rows if all of its column is zero
            df_bands.append(df)
        final_df_per_bands = pd.concat(df_bands, axis=1) 
        final_df_per_bands["Landcover"] = roi_name
        final_df_per_bands.to_csv(f"csv_files\{roi_name}.csv", index = False)
        print(final_df_per_bands)

Converting raster array to DataFrame with column name BLUE
Converting raster array to DataFrame with column name GREEN
Converting raster array to DataFrame with column name RED
Converting raster array to DataFrame with column name NIR
Converting raster array to DataFrame with column name NDVI
          BLUE   GREEN     RED     NIR      NDVI Landcover
1557    0.2186  0.2715  0.2688  0.2799  0.107937   builtup
1558    0.2206  0.2830  0.2684  0.2716  0.123908   builtup
1559    0.2318  0.2909  0.2921  0.2879  0.113067   builtup
1560    0.2424  0.3219  0.3152  0.3120  0.140883   builtup
1561    0.2363  0.2990  0.2931  0.3011  0.117131   builtup
...        ...     ...     ...     ...       ...       ...
815362  0.1539  0.1732  0.1876  0.1940  0.059003   builtup
815363  0.1378  0.1577  0.1784  0.1842  0.067344   builtup
815364  0.1327  0.1503  0.1688  0.1768  0.062191   builtup
815365  0.1277  0.1435  0.1580  0.1723  0.058260   builtup
815366  0.1216  0.1344  0.1450  0.1604  0.050000   builtu

# CREATE THE TRAINING DF THAT CONSIST OF ALL THE DATA FROM ROIs

In [5]:
df_roi = []
for roi_csv_path in glob.glob(os.path.join("csv_files", "*.csv")):
    roi_csv_name = os.path.splitext(os.path.basename(roi_csv_path))[0]
    if roi_csv_name != "dataset":
        df = pd.read_csv(roi_csv_path)
        df_roi.append(df)
final_training_data = pd.concat(df_roi, axis=0)
final_training_data.to_csv("csv_files/dataset.csv", index=False)

END